[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/01_Multimodal_Foundations/06_tokenization_embeddings/06_tokenization_embeddings.ipynb)

# 06. Tokenization & Embeddings

**This notebook covers:**
- BPE tokenizer from scratch
- Patch embedding for ViT
- Comparison: WordPiece vs BPE vs SentencePiece
- Vocabulary analysis and coverage

**Runtime:** ~10–15 minutes on CPU

---

> **Theory & derivations:** See [README.md](./README.md) for full step-by-step math.


In [ ]:
# ============================================================
#  Google Colab Setup — Run this cell FIRST
# ============================================================
import os, sys

try:
    import google.colab
    IN_COLAB = True
    print("Google Colab detected — setting up environment...")
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git"
    REPO_DIR = "/content/Multimodal-Deep-Learning"

    if not os.path.exists(REPO_DIR):
        print("Cloning repository...")
        !git clone --depth 1 {REPO_URL} {REPO_DIR}
    else:
        print("Repository already cloned")

    print("Installing dependencies...")
    !pip install -q -r {REPO_DIR}/requirements.txt

    MODULE_DIR = f"{REPO_DIR}/01_Multimodal_Foundations/06_tokenization_embeddings"
    os.chdir(MODULE_DIR)
    os.makedirs(f"{REPO_DIR}/assets", exist_ok=True)

    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)

    print(f"Colab setup complete — {os.getcwd()}")

    import torch
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("Device: CPU (all notebooks work fine on CPU)")
else:
    os.makedirs("../../assets", exist_ok=True)
    print("Running locally — all set!")

In [ ]:
import sys
sys.path.append('../..')

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

try:
    from utils.visualization import set_style
    from utils.helpers import count_parameters, get_device
    set_style()
except ImportError:
    def set_style():
        plt.rcParams.update({'figure.figsize': (10, 6), 'figure.dpi': 100})
    def count_parameters(model):
        total = sum(p.numel() for p in model.parameters())
        print(f"Total parameters: {total:,}")
        return total
    def get_device():
        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    set_style()

torch.manual_seed(42)
np.random.seed(42)
device = get_device() if callable(get_device) else torch.device('cpu')
print(f"PyTorch {torch.__version__} | Device: {device}")

## 1. BPE Tokenizer From Scratch

Byte Pair Encoding iteratively merges the most frequent symbol pairs.


In [ ]:
from collections import Counter, defaultdict

corpus = [
    "low lower lowest",
    "new newer newest",
    "wide wider widest",
    "a cat sits on the mat",
    "the cat and the dog",
] * 5
words = [w for text in corpus for w in text.split()]
vocab = set("".join(words))

def get_pairs(word):
    pairs = Counter()
    for i in range(len(word) - 1):
        pairs[(word[i], word[i+1])] += 1
    return pairs

# Word -> list of chars
word_splits = {w: list(w) + ["</w>"] for w in set(words)}

def merge_pair(pair, splits):
    new_splits = {}
    bigram = " ".join(pair)
    replacement = "".join(pair)
    for word, symbols in splits.items():
        s = " ".join(symbols)
        s = s.replace(bigram, replacement)
        new_splits[word] = s.split()
    return new_splits

merges = []
splits = word_splits.copy()
for step in range(8):
    pairs = Counter()
    for word, syms in splits.items():
        for p in get_pairs("".join(syms).replace("</w>", "")):
            pairs[p] += corpus.count(" ".join(word for word in words if word == word)) or 1
    for word, syms in splits.items():
        w = "".join(syms).replace("</w>", "")
        for i in range(len(w)-1):
            pairs[(w[i], w[i+1])] += words.count(word)
    if not pairs:
        break
    best = pairs.most_common(1)[0][0]
    merges.append(best)
    splits = merge_pair(best, splits)
    print(f"Merge {step+1}: {best} -> {''.join(best)}")

print("\nFinal merges:", merges[:5])

## 2. Encode / Decode With Learned Merges


In [ ]:
def bpe_encode(word, merges):
    symbols = list(word) + ["</w>"]
    for a, b in merges:
        i = 0
        while i < len(symbols) - 1:
            if symbols[i] == a and symbols[i+1] == b:
                symbols = symbols[:i] + [a+b] + symbols[i+2:]
            else:
                i += 1
    return symbols

sample = "lowest"
encoded = bpe_encode(sample, merges)
print(f"BPE('{sample}') = {encoded}")

## 3. Patch Embedding for ViT

Images become sequences: flatten $P \times P$ patches and linearly project to $D$.


In [ ]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_ch=3, d_model=128):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_ch, d_model, kernel_size=patch_size, stride=patch_size)
        self.cls = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos = nn.Parameter(torch.randn(1, self.n_patches + 1, d_model) * 0.02)

    def forward(self, x):
        x = self.proj(x).flatten(2).transpose(1, 2)
        cls = self.cls.expand(x.size(0), -1, -1)
        x = torch.cat([cls, x], dim=1) + self.pos
        return x

patch_embed = PatchEmbedding()
img = torch.randn(2, 3, 32, 32)
tokens = patch_embed(img)
print(f"Image {img.shape} -> patch tokens {tokens.shape}  (1 CLS + {patch_embed.n_patches} patches)")
count_parameters(patch_embed)

## 4. Tokenizer Comparison (HuggingFace)


In [ ]:
try:
    from transformers import BertTokenizer, GPT2Tokenizer
    sample = "Multimodal models align vision and language."

    wp = BertTokenizer.from_pretrained("bert-base-uncased")
    bpe = GPT2Tokenizer.from_pretrained("gpt2")

    wp_ids = wp.encode(sample)
    bpe_ids = bpe.encode(sample)

    print("WordPiece (BERT):", wp.tokenize(sample), "->", len(wp_ids), "ids")
    print("BPE (GPT-2):     ", bpe.tokenize(sample), "->", len(bpe_ids), "ids")
except Exception as e:
    print("Skipping HF comparison (offline):", e)
    print("WordPiece splits rare words (mult ##im ##odal); BPE uses byte-level merges.")

## 5. Vocabulary Analysis


In [ ]:
word_freq = Counter(words)
top = word_freq.most_common(15)
ranks, counts = zip(*top)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar([w for w, _ in top], counts, color='steelblue')
axes[0].set_title('Top Token Frequencies'); axes[0].tick_params(axis='x', rotation=45)

# Zipf-style log-log
all_counts = sorted(word_freq.values(), reverse=True)
axes[1].loglog(range(1, len(all_counts)+1), all_counts, '.')
axes[1].set_xlabel('Rank'); axes[1].set_ylabel('Frequency'); axes[1].set_title('Zipf Distribution')
plt.tight_layout(); plt.show()

coverage = sum(c for _, c in top) / sum(word_freq.values())
print(f"Top-15 tokens cover {coverage*100:.1f}% of corpus occurrences")

## Summary

Built BPE merges, ViT patch embedding, and analyzed vocabulary statistics.

**Next:** Module 02 — CLIP from scratch
